### Week 3 Exercise - Synthetic Review Generator

The goal of this exercise is to generate synthetic reviews across a wide range of emotions which can be used as AI training data.


This makes use of background threads for performance as inspired by - https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D#scrollTo=C9zvDGWD5pKp

In [ ]:
# Pip install deps
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 gradio

In [24]:
# imports

from threading import Thread
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer, BitsAndBytesConfig
import torch
import gradio as gr

# Constants
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

# Log into HF
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
# Quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
def build_user_prompt(review_count):
    return f"""
You generate synthetic product reviews for a sentiment-classification dataset. Generate exactly {review_count} reviews.

For each review, produce these fields:

- product_category: A broad product type, such as electronics, kitchen, footwear, or furniture.
- review_text: One to four sentences that sound like something a real customer would write. Natural imperfections in grammar or tone are fine.
- sentiment: One of "positive", "negative", or "neutral".
- intensity: An integer from 1 to 5. Use a low number for mild language and a high number for strong language.
- emotion: One of "satisfaction", "frustration", "anger", "disappointment", "excitement", "gratitude", "indifference", "regret", "relief", or "confusion".

Vary the product categories from one review to the next. Do not repeat the same product twice in a row.

Keep the label choices consistent with the review. A negative review with an intensity of 5 should sound angry or distressed. An intensity-1 review should sound fairly flat or indifferent. The emotion should also make sense for the sentiment and intensity. Do not use "gratitude" with a negative sentiment or "indifference" with an intensity of 5.

Give each review specific details about using the product, such as its fit, materials, battery life, packaging, delivery, durability, or ease of use. Avoid generic praise and complaints that could apply to anything.

Return the {review_count} reviews as a single markdown table with exactly these columns, in this order:
Product Category | Review Text | Sentiment | Intensity | Emotion

Use this exact header row followed by the separator row, then one row per review:
| Product Category | Review Text | Sentiment | Intensity | Emotion |
|---|---|---|---|---|

Return the table only. Do not add a title, summary, code fences, or any text before or after the table.
"""


In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",
    quantization_config=quant_config,
)

In [ ]:
def generate_reviews(review_count):
    review_count = int(review_count)
    messages = [{"role": "user", "content": build_user_prompt(review_count)}]

    # The prompt tensor must be on the same device as the model.
    input_ids = tokenizer.apply_chat_template(
        messages,
        # Keep the assistant role header in the prompt instead of the streamed response.
        add_generation_prompt=True,
        return_tensors="pt",
    ).to('cuda')

    # The streamer turns generated token IDs into text as they arrive.
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,
        skip_special_tokens=True,
        timeout=60.0,
    )
    # Allow more output tokens when the user asks for more reviews.
    generation_kwargs = {
        "input_ids": input_ids,
        "streamer": streamer,
        "max_new_tokens": 250 + (review_count * 150),
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "pad_token_id": tokenizer.eos_token_id,
    }

    # model.generate() blocks, so run it separately while this function reads the stream.
    generation_thread = Thread(
        target=model.generate,
        kwargs=generation_kwargs,
        daemon=True,
    )
    generation_thread.start()

    # Gradio replaces the output on each yield, so send the full response built so far.
    response = ""
    for text in streamer:
        response += text
        yield response

    generation_thread.join()

In [ ]:
# Gradio interface
with gr.Blocks() as demo:
    gr.Markdown("# Synthetic Review Generator")
    gr.Markdown("Choose how many reviews to create.")

    review_count = gr.Slider(
        minimum=1,
        maximum=20,
        value=5,
        step=1,
        label="Number of reviews",
    )
    generate_button = gr.Button("Generate reviews", variant="primary")
    output = gr.Markdown(label="Generated reviews", min_height=200)

    generate_button.click(
        fn=generate_reviews,
        inputs=review_count,
        outputs=output,
    )

demo.queue().launch()